In [ ]:
from stratified_gm import run_analysis
import cebra_dim_reduction
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import umap
from itertools import combinations
from numpy.typing import ArrayLike, NDArray
from pathlib import Path
from matplotlib.colors import Normalize
import numpy as np
import plotly.graph_objects as go
import ot
import hdbscan
from scipy.spatial.transform import Rotation

In [2]:

def sample_sphere_surface(N: int, n: int, d: int = 3, rng=None) -> np.ndarray:
    """
    Generate N collections of n iid samples uniformly from the surface
    of the unit sphere S^(d-1) in R^d.

    Returns
    -------
    samples : ndarray, shape (N, n, d)
        samples[i, j] is one point on the sphere.
    """
    rng = np.random.default_rng(rng)

    # Draw standard normal vectors
    x = rng.normal(size=(N, n, d))

    # Normalize each vector to length 1
    norms = np.linalg.norm(x, axis=-1, keepdims=True)
    samples = x / norms

    return samples



def sample_torus_surface(
    N: int,
    n: int,
    R: float = 2.0,
    r: float = 1.0,
    rng=None,
) -> np.ndarray:
    """
    Generate N collections of n iid samples uniformly from the surface
    of a torus in R^3.

    The torus has major radius R and minor radius r, with R > r > 0.

    Returns
    -------
    samples : ndarray, shape (N, n, 3)
        samples[i, j] is one point on the torus surface.
    """
    if not (R > r > 0):
        raise ValueError("Require R > r > 0.")

    rng = np.random.default_rng(rng)

    total = N * n

    # u is uniform around the main circle
    u = rng.uniform(0, 2 * np.pi, size=total)

    # v is NOT uniform for uniform surface area.
    # Its density is proportional to R + r cos(v).
    v_samples = []

    while sum(len(chunk) for chunk in v_samples) < total:
        m = total - sum(len(chunk) for chunk in v_samples)

        v_candidate = rng.uniform(0, 2 * np.pi, size=2 * m)
        accept_prob = (R + r * np.cos(v_candidate)) / (R + r)

        accepted = v_candidate[rng.uniform(size=2 * m) < accept_prob]
        v_samples.append(accepted)

    v = np.concatenate(v_samples)[:total]

    x = (R + r * np.cos(v)) * np.cos(u)
    y = (R + r * np.cos(v)) * np.sin(u)
    z = r * np.sin(v)

    samples = np.stack([x, y, z], axis=-1)
    return samples.reshape(N, n, 3)




def sample_cube_surface(
    N: int,
    n: int,
    side_length: float = 2.0,
    rng=None,
) -> np.ndarray:
    """
    Generate N collections of n iid samples uniformly from the surface
    of a cube centered at the origin.

    Returns
    -------
    samples : ndarray, shape (N, n, 3)
        samples[i, j] is one point on the cube surface.
    """
    if side_length <= 0:
        raise ValueError("side_length must be positive.")

    rng = np.random.default_rng(rng)

    total = N * n
    half = side_length / 2

    # Pick one of the 6 faces uniformly
    faces = rng.integers(0, 6, size=total)

    # Pick two coordinates uniformly on the chosen face
    coords = rng.uniform(-half, half, size=(total, 3))

    # Set one coordinate to +/- half depending on the face
    axis = faces // 2          # 0 for x, 1 for y, 2 for z
    sign = 2 * (faces % 2) - 1 # -1 or +1

    coords[np.arange(total), axis] = sign * half

    return coords.reshape(N, n, 3)


def sample_triangle_uniform(a, b, c, m: int, rng) -> np.ndarray:
    """
    Uniformly sample m points from the triangle with vertices a, b, c.
    """
    u = rng.uniform(size=m)
    v = rng.uniform(size=m)

    # Reflect points outside the unit simplex
    mask = u + v > 1
    u[mask] = 1 - u[mask]
    v[mask] = 1 - v[mask]

    return a + u[:, None] * (b - a) + v[:, None] * (c - a)


def sample_tetrahedron_surface(
    N: int,
    n: int,
    side_length: float = 2.0,
    rng=None,
) -> np.ndarray:
    """
    Generate N collections of n iid samples uniformly from the surface
    of a regular triangular pyramid, i.e. a regular tetrahedron.

    Returns
    -------
    samples : ndarray, shape (N, n, 3)
    """
    if side_length <= 0:
        raise ValueError("side_length must be positive.")

    rng = np.random.default_rng(rng)
    total = N * n

    # Vertices of a regular tetrahedron centered at the origin.
    vertices = np.array([
        [1,  1,  1],
        [1, -1, -1],
        [-1, 1, -1],
        [-1, -1, 1],
    ], dtype=float)

    # Scale to desired side length.
    # The original side length is 2 * sqrt(2).
    vertices *= side_length / (2 * np.sqrt(2))

    faces = np.array([
        [0, 1, 2],
        [0, 1, 3],
        [0, 2, 3],
        [1, 2, 3],
    ])

    # For a regular tetrahedron, all 4 faces have equal area.
    chosen_faces = rng.integers(0, 4, size=total)

    samples = np.empty((total, 3))

    for f in range(4):
        idx = np.where(chosen_faces == f)[0]
        if len(idx) == 0:
            continue

        a, b, c = vertices[faces[f]]
        samples[idx] = sample_triangle_uniform(a, b, c, len(idx), rng)

    return samples.reshape(N, n, 3)

def sample_three_surfaces(
    N: int,
    n: int,
    torus_R: float = 2.0,
    torus_r: float = 1.0,
    cube_side_length: float = 2.0,
    rng=None,
) -> np.ndarray:
    """
    Generate N collections of n samples from each of:
    sphere surface, torus surface, cube surface.

    Returns
    -------
    samples : ndarray, shape (3 * N, n, 3)

        samples[0:N]       are sphere samples
        samples[N:2*N]     are torus samples
        samples[2*N:3*N]   are cube samples
    """
    rng = np.random.default_rng(rng)

    sphere = sample_sphere_surface(N, n, rng=rng)
    torus = sample_torus_surface(N, n, R=torus_R, r=torus_r, rng=rng)
    cube = sample_cube_surface(N, n, side_length=cube_side_length, rng=rng)

    return np.concatenate([sphere, torus, cube], axis=0)


def apply_random_rotations(samples: np.ndarray, rng=None) -> np.ndarray:
    """
    Apply a different random SO(3) rotation to each sample group.

    Parameters
    ----------
    samples : ndarray, shape (num_groups, n, 3)

    Returns
    -------
    rotated : ndarray, shape (num_groups, n, 3)
    """
    if samples.ndim != 3 or samples.shape[-1] != 3:
        raise ValueError("samples must have shape (num_groups, n, 3).")

    rng = np.random.default_rng(rng)

    num_groups = samples.shape[0]

    rotations = Rotation.random(num_groups, random_state=rng)
    R = rotations.as_matrix()  # shape (num_groups, 3, 3)

    # For each group g and point i:
    # rotated[g, i] = R[g] @ samples[g, i]
    rotated = np.einsum("gij,gnj->gni", R, samples)

    return rotated



def sample_four_surfaces(
    N: int,
    n: int,
    torus_R: float = 2.0,
    torus_r: float = 1.0,
    cube_side_length: float = 2.0,
    tetra_side_length: float = 2.0,
    random_rotate: bool = True,
    rng=None,
) -> np.ndarray:
    """
    Generate N collections of n samples from each of 4 surfaces:

    1. sphere
    2. torus
    3. cube
    4. triangular pyramid / tetrahedron

    Optionally applies a different random SO(3) rotation to every group.

    Returns
    -------
    samples : ndarray, shape (4 * N, n, 3)
    """
    rng = np.random.default_rng(rng)

    sphere = sample_sphere_surface(N, n, rng=rng)

    torus = sample_torus_surface(
        N,
        n,
        R=torus_R,
        r=torus_r,
        rng=rng,
    )

    cube = sample_cube_surface(
        N,
        n,
        side_length=cube_side_length,
        rng=rng,
    )

    tetra = sample_tetrahedron_surface(
        N,
        n,
        side_length=tetra_side_length,
        rng=rng,
    )

    samples = np.concatenate([sphere, torus, cube, tetra], axis=0)

    if random_rotate:
        samples = apply_random_rotations(samples, rng=rng)

    return samples



In [4]:
test_points_shapes = sample_four_surfaces(25, 1000) 
test_results_shape = run_analysis(test_points_shapes, './/random_tests//shapes_mmd.png')

In [5]:

def sample_point_subsets(
    points: ArrayLike,
    k: int,
    subset_size: int,
    *,
    replace: bool = False,
    seed: int | None = None,
) -> NDArray:
    """
    Sample k subsets of points, each containing subset_size points.

    Parameters
    ----------
    points:
        Array with shape (num_points, num_features).
    k:
        Number of subsets to generate.
    subset_size:
        Number of points in each subset.
    replace:
        If True, points are sampled IID with replacement.
        If False, points are unique within each subset.
    seed:
        Random seed for reproducibility.

    Returns
    -------
    subsets:
        Array with shape (k, subset_size, num_features).
    """
    points = np.asarray(points)

    if points.ndim < 2:
        raise ValueError("points must have shape (num_points, num_features).")
    if k <= 0 or subset_size <= 0:
        raise ValueError("k and subset_size must be positive integers.")
    if not replace and subset_size > len(points):
        raise ValueError(
            "subset_size cannot exceed the number of points "
            "when replace=False."
        )

    rng = np.random.default_rng(seed)

    if replace:
        indices = rng.integers(
            low=0,
            high=len(points),
            size=(k, subset_size),
        )
    else:
        indices = np.stack([
            rng.choice(len(points), size=subset_size, replace=False)
            for _ in range(k)
        ])

    return points[indices]

In [6]:
time3_embeddings_values = []
posdir3_embeddings_values = []

for i in range(4):
    A = np.load(f".//Consistency_test_data//time3_embedding_values_rat_{i}.npy")
    B = np.load(f".//Consistency_test_data//posdir3_embedding_values_rat_{i}.npy")
    subsets_A = sample_point_subsets(A, 20, 1000)
    subsets_B = sample_point_subsets(B, 20, 1000)

    for subset in subsets_A:
        time3_embeddings_values.append(subset)
    for subset in subsets_B:
        posdir3_embeddings_values.append(subset)

In [7]:
results_time3 = run_analysis(time3_embeddings_values, './/random_tests//time3_mmd.png')
results_posdir3 = run_analysis(posdir3_embeddings_values, './/random_tests//posdir3_mmd.png')